# Assignment 1 – Gymnasium, Monte Carlo and TD(0)

**Reinforcement Learning – Homework 1, Jaume Manero**

This notebook covers the five activities of HW1:

1. Familiarisation with Gym environments (CartPole, Taxi, LunarLander, FrozenLake).
2. Monte Carlo on FrozenLake 4x4 and 8x8.
3. Monte Carlo on the Volcano custom environment.
4. Monte Carlo on Taxi.
5. TD(0) on Taxi and FrozenLake.

And answers the five questions of Section 5 of the spec.

All training code lives in `hw1_frozen_lake/`:

- `monte_carlo.py` – first-visit MC for FrozenLake (used in Activity 2).
- `mc_generic.py` – generic MC control (used in Activities 3 and 4).
- `td_learning.py` – TD(0) prediction and TD-control (used in Activity 5).
- `volcano_world.py` – Volcano environment (vendored from `castorgit/RL_course`).
- `run_full.py` – trains everything and dumps to `results_full/`.

Reproducing the artifacts:

```bash
python -m hw1_frozen_lake.run_full --seed 42
```

Takes ~5–8 min on a modest CPU. The notebook below loads the resulting `.npz` files.

In [ ]:
import sys, os
# Make the repo root importable when running the notebook from inside hw1_frozen_lake/
if os.path.basename(os.getcwd()) == 'hw1_frozen_lake':
    sys.path.insert(0, os.path.abspath('..'))

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym

RESULTS = Path('hw1_frozen_lake/results_full')
if not RESULTS.exists():
    RESULTS = Path('results_full')  # in case running notebook from inside hw1_frozen_lake/
print('Loading artifacts from:', RESULTS.resolve() if RESULTS.exists() else '<not found, run run_full.py first>')

## Activity 1 – Familiarisation with Gym environments

We compare four environments. For each one we report observation / action space, reward range, episode termination and rendering modes.

In [ ]:
def describe(env_id, **kwargs):
    env = gym.make(env_id, **kwargs)
    obs, info = env.reset(seed=0)
    print(f'--- {env_id} ---')
    print(f'Observation space : {env.observation_space}')
    print(f'Action space      : {env.action_space}')
    print(f'Reward range      : {getattr(env, "reward_range", (None, None))}')
    print(f'Sample obs        : {obs}')
    sample_action = env.action_space.sample()
    obs, reward, terminated, truncated, _ = env.step(sample_action)
    print(f'After random act  : action={sample_action}, reward={reward}, terminated={terminated}')
    print()

describe('FrozenLake-v1', map_name='4x4', is_slippery=False)
describe('FrozenLake-v1', map_name='8x8', is_slippery=True)
try:
    describe('Taxi-v4')
except Exception:
    describe('Taxi-v3')
describe('CartPole-v1')
try:
    describe('LunarLander-v3')
except Exception:
    describe('LunarLander-v2')


### Comparative description

| Env | Observation | Action | Reward | Discrete? | Comments |
|---|---|---|---|---|---|
| FrozenLake-v1 | Discrete(16) [4x4] / Discrete(64) [8x8] | Discrete(4) | 0 or 1 | yes | Sparse reward, slippery dynamics make it stochastic. |
| Taxi-v3/v4 | Discrete(500) | Discrete(6) | dense (-10..+20) | yes | Larger state space, multi-step tasks (pick + drop). |
| CartPole-v1 | Box(4) | Discrete(2) | +1 per step | continuous obs | Continuous physics, dense reward, terminal on fall. |
| LunarLander-v3 | Box(8) | Discrete(4) | dense, large range | continuous obs | High-dim, sparse-but-shaped reward, requires generalisation (no tabular methods). |

**Main differences:** FrozenLake and Taxi are *toy text* envs with discrete obs and tabular methods work directly. CartPole and LunarLander are *classic control* / *Box2D* envs with continuous observations: tabular methods need discretisation (HW3) or function approximation (HW4).

## Activity 2 – Monte Carlo on FrozenLake 4x4 and 8x8

We use first-visit MC control with ε-greedy exploration. The implementation is in `hw1_frozen_lake/monte_carlo.py` (see `mc_control`).

Key update rule:

$$Q(s, a) \leftarrow Q(s, a) + \frac{1}{N(s,a)}\,(G_t - Q(s, a))$$

(incremental sample-mean form, equivalent to the constant-α update with α = 1/N).

In [ ]:
def show_returns(npz_path, title):
    data = np.load(npz_path)
    fig, ax = plt.subplots(figsize=(7, 3.5))
    for key, label in [('returns_mc', 'MC first-visit'), ('returns_td', 'TD-control')]:
        if key in data.files:
            r = data[key]
            window = max(100, len(r) // 50)
            ma = np.convolve(r, np.ones(window)/window, mode='valid')
            ax.plot(np.arange(len(ma)) + window - 1, ma, label=label)
    ax.set_title(title); ax.set_xlabel('Episode'); ax.set_ylabel('Return (MA)'); ax.legend(); ax.grid(alpha=.3)
    plt.show()

def show_value_grid(Q, shape, title):
    V = Q.max(axis=1).reshape(shape)
    fig, ax = plt.subplots(figsize=(shape[1]*0.6+1, shape[0]*0.6+1))
    im = ax.imshow(V, cmap='viridis')
    for i in range(shape[0]):
        for j in range(shape[1]):
            ax.text(j, i, f'{V[i,j]:.2f}', ha='center', va='center', fontsize=8,
                    color='white' if V[i,j] < V.max()*0.6 else 'black')
    ax.set_title(title); fig.colorbar(im, ax=ax, fraction=0.04); plt.show()

ARROWS_FL = {0: '<', 1: 'v', 2: '>', 3: '^'}
def render_policy(Q, shape, arrows=ARROWS_FL):
    pol = Q.argmax(axis=1).reshape(shape)
    return '\n'.join(' '.join(arrows[int(a)] for a in row) for row in pol)

show_returns(RESULTS / 'frozenlake_4x4_deterministic.npz', 'FrozenLake 4x4 deterministic')
show_returns(RESULTS / 'frozenlake_4x4_slippery.npz', 'FrozenLake 4x4 SLIPPERY')
show_returns(RESULTS / 'frozenlake_8x8_deterministic.npz', 'FrozenLake 8x8 deterministic')

In [ ]:
data = np.load(RESULTS / 'frozenlake_4x4_deterministic.npz')
show_value_grid(data['Q_mc'], (4, 4), 'V_MC – FrozenLake 4x4 deterministic')
print('Greedy policy (MC):')
print(render_policy(data['Q_mc'], (4, 4)))

data = np.load(RESULTS / 'frozenlake_8x8_deterministic.npz')
show_value_grid(data['Q_mc'], (8, 8), 'V_MC – FrozenLake 8x8 deterministic')
print('Greedy policy (MC):')
print(render_policy(data['Q_mc'], (8, 8)))

## Activity 3 – Monte Carlo on Volcano

The Volcano world is the 3×4 GridWorld defined in `volcano_world.py`. With slippery dynamics (1/3 left, 1/3 intended, 1/3 right) the optimal policy emerges as **risk-averse**: it heads to the safe goal `G` (+2) instead of the risky `V` (+20) because the lava cells are too close.

```
 . . L V       L = lava (-50, terminal)
 S . L .       V = view (+20, terminal, risky)
 G . . .       G = safe (+2,  terminal, conservative)
```

In [ ]:
data = np.load(RESULTS / 'volcano.npz')
show_value_grid(data['Q'], (3, 4), 'V_MC – Volcano (slippery)')
ARROWS_VOL = {0: '^', 1: '>', 2: 'v', 3: '<'}  # Volcano: 0=N,1=E,2=S,3=W
print('Greedy policy (MC) – Volcano:')
print(render_policy(data['Q'], (3, 4), arrows=ARROWS_VOL))
print(f'\nLast 100 training episodes mean return: {data["returns"][-100:].mean():.3f}')

**Discusión.** El agente aprende que la diferencia entre el +20 de la vista y el +2 de la salida segura no compensa el riesgo de caer en la lava (-50) cuando hay 1/3 de probabilidad de resbalar. La política óptima va al sur (G) en lugar de al noreste (V) – es la *aversión al riesgo emergente* que vimos en clase: la ecuación de Bellman penaliza el valor de los estados cercanos a la lava por la probabilidad de transición.

## Activity 4 – Monte Carlo on Taxi

Taxi has |S|=500 estados (taxi position × 5 passenger locations × 4 destinations) y |A|=6 acciones (N, S, E, W, pickup, dropoff). El reward es:
- −1 por paso
- +20 por dropoff exitoso
- −10 por pickup/dropoff ilegal

In [ ]:
data = np.load(RESULTS / 'taxi.npz')
show_returns(RESULTS / 'taxi.npz', 'Taxi – MC vs TD')
print(f'MC last-100 train return: {data["returns_mc"][-100:].mean():.2f}')
print(f'TD last-100 train return: {data["returns_td"][-100:].mean():.2f}')
print(f'Q-table shape: {data["Q_mc"].shape}')

## Activity 5 – TD(0) on FrozenLake/Taxi

El TD-control (Q-Learning con la regla TD(0) one-step) está implementado en `td_learning.py`. La actualización es:

$$Q(s, a) \leftarrow Q(s, a) + \alpha\,[\,r + \gamma \max_{a'} Q(s', a') - Q(s, a)\,]$$

A diferencia de MC, TD actualiza **a cada paso** usando bootstrapping del siguiente estado. Comparamos curvas de aprendizaje y políticas finales con MC.

In [ ]:
# Side-by-side learning curves – slippery FrozenLake (the interesting case)
data = np.load(RESULTS / 'frozenlake_4x4_slippery.npz')
fig, ax = plt.subplots(figsize=(8, 4))
for key, label in [('returns_mc', 'MC first-visit'), ('returns_td', 'TD-control')]:
    r = data[key]
    window = max(100, len(r) // 50)
    ma = np.convolve(r, np.ones(window)/window, mode='valid')
    ax.plot(np.arange(len(ma)) + window - 1, ma, label=label)
ax.set_title('FrozenLake 4x4 SLIPPERY – MC vs TD')
ax.set_xlabel('Episode'); ax.set_ylabel('Success rate (MA)'); ax.grid(alpha=.3); ax.legend()
plt.show()

show_value_grid(data['Q_mc'], (4, 4), 'V_MC – slippery')
show_value_grid(data['Q_td'], (4, 4), 'V_TD – slippery')

## Comparative table (Activities 2-5)

Todos los entornos, episodios, tiempo de entrenamiento, y rendimiento (greedy eval sobre 1000 episodios).

In [ ]:
from IPython.display import Markdown
table_md = (RESULTS / 'comparative_table.md').read_text()
Markdown(table_md)

## Section 5 – Questions

### Q1. Comparative table between methods and environments

Ver la tabla anterior. Resumen cualitativo:
- **FrozenLake 4x4 deterministic**: trivial. Ambos métodos llegan al 100% de éxito en pocos miles de episodios.
- **FrozenLake 4x4 slippery**: TD claramente más rápido y con mejor rendimiento. MC sufre porque la varianza del retorno bajo dinámica resbaladiza es alta.
- **FrozenLake 8x8 deterministic**: ambos llegan a 100%, pero requieren más episodios (estado más grande, recompensa muy diferida).
- **Volcano**: MC encuentra la política aversa al riesgo (sale por la salida segura).
- **Taxi**: TD aprende mucho antes que MC. Para 20k episodios, TD ya está cerca del óptimo (~+8) mientras MC sigue subiendo.

### Q2. ¿Qué método converge mejor en mis experimentos?

**TD(0)/Q-Learning** converge mejor en todos los entornos no triviales. Razones:
- TD actualiza a cada paso → aprovecha trayectorias parciales.
- Bootstrapping reduce la varianza (introduce sesgo, pero menor varianza compensa con creces).
- MC sólo actualiza al final del episodio: en entornos largos (Taxi) o con alta varianza (FrozenLake slippery) MC paga un coste enorme en episodios.

### Q3. `is_slippery=True` en FrozenLake. ¿Qué pasa? ¿Por qué hace el problema más difícil?

Con `is_slippery=True` cada acción tiene 1/3 de probabilidad de ejecutarse, 1/3 de resbalar a la izquierda y 1/3 de resbalar a la derecha (perpendicular). Esto:
1. Convierte un MDP determinista en estocástico → la política óptima ya no es "ir directo a la meta" sino una que **evita estados de los que se puede caer al hoyo**.
2. El retorno por episodio tiene altísima varianza → MC necesita muchísimos más episodios para una buena estimación.
3. La tasa de éxito óptima cae de 100% a ~74% (el azar pone un techo a lo que el agente puede lograr).

### Q4. Comparación FrozenLake 4x4 vs 8x8

| | 4x4 | 8x8 |
|---|---|---|
| Estados | 16 | 64 |
| Episodios para converger | ~30k | ~120k |
| Tiempo CPU | <1s | varios segundos |

Aumentar el tamaño de la rejilla:
- Multiplica el espacio de estados por 4.
- La recompensa está más diferida (caminos más largos hasta la meta).
- ε-greedy con la misma tasa de exploración tarda más en visitar todos los estados.

### Q5. FrozenLake vs Taxi – ¿cuál es más difícil para MC?

**Taxi es más difícil para MC**, no por número de estados sino por **estructura de la tarea**:
- Taxi tiene 500 estados pero exige una secuencia precisa: navegar → pickup → navegar → dropoff. Un episodio sin pickup correcto da −10 al final, lo que envenena MC.
- FrozenLake tiene recompensa binaria (0 o 1) y los estados malos son terminales: no hay "crédito temporal mal asignado" como en Taxi.
- En MC el problema del *credit assignment* (saber qué acción anterior causó la recompensa actual) es severo en Taxi.
- TD soluciona esto mucho mejor porque propaga el valor estado a estado en lugar de esperar al final del episodio.

## Notas finales

- Todo el código vive en `hw1_frozen_lake/`. El driver `run_full.py` regenera todos los artefactos (`results_full/`).
- Seeds usadas: 42. Todas las curvas son medias móviles para suavizar.
- **AI policy disclosure**: La estructura del código y los algoritmos centrales (loops MC y TD) se han escrito a mano siguiendo Sutton & Barto Cap. 5 y 6. La asistencia de IA se ha limitado a refactor menor (typing, plotting helpers) y a esta redacción del notebook.